# Local grid refinement with MODFLOW 6

**What this notebook covers.** Build a MODFLOW 6 model that uses **local grid refinement (LGR)** to resolve a small area of interest at higher resolution than the rest of the domain. You will mark a refinement block in a coarse "parent" grid, generate a finer "child" grid for it, connect the two with a groundwater flow (GWF) exchange, run the coupled simulation, and compare heads across the shared boundary.

The domain is the simple three-layer groundwater system shown below.

![lgr_example](../data/lgr_example.png)

## Imports

Import FloPy, NumPy, and Matplotlib, and print the package versions so the run is reproducible.

In [ ]:
%matplotlib inline
import sys

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from flopy.utils.lgrutil import Lgr

print(sys.version)
print(f"python executable: {sys.executable}")
print(f"numpy version: {np.__version__}")
print(f"matplotlib version: {mpl.__version__}")
print(f"flopy version: {flopy.__version__}")

## Assign problem variables

Set the parent model's grid dimensions, hydraulic properties, and stresses. Lengths are in feet — keep every input in consistent units. Build the river stress-period data with a list comprehension: one river cell in the last column (`ncol - 1`) of every row.

```python
# ((layer, row, column), stage, conductance, rbot)
((0, ir, ncol - 1), 320.0, 1.0e5, 318.0)
```

In [ ]:
# model info
model_name = "parent"
model_ws = "./models/lgr-flopy"

# grid properties
nlay = 3
nrow = 21
ncol = 20
delr = 500.0
delc = 500.0
top = 400.0
botm = [220.0, 200.0, 0.0]

# hydraulic properties
hk0 = 50.0
vk0 = 10.0
hk1 = 0.01
vk1 = 0.01
hk2 = 200.0
vk2 = 20.0

# stresses
well_discharge = -1.5e5
rech = 0.005
rivspd = [[(0, ir, ncol - 1), 320.0, 1.0e5, 318.0] for ir in range(nrow)]

## Mark where the child grid goes

Create an integer array (**IDOMAIN**) for the parent grid: `1` keeps a cell active, `0` marks the block the finer child grid will replace. Here the block is carved from all three layers.

In [ ]:
idomain = np.ones((nlay, nrow, ncol), dtype=int)
idomain[0:3, 8:14, 7:13] = 0
idomain[0:3, 7, 8:10] = 0
plt.matshow(idomain[0])

Create a FloPy `Lgr` helper for the marked block. It derives everything the child model needs — its grid shape, cell sizes, top and bottom elevations, lower-left origin, IDOMAIN, and the parent–child **exchange data** (the connected cell faces that let water flow between the two grids). `ncpp = 3` splits each parent cell into a 3×3 block of child cells; `ncppl = [1, 1, 1]` keeps one child layer per parent layer.

In [ ]:
# create the flopy Lgr object to help define the child model

ncpp = 3
ncppl = [1, 1, 1]
lgr = Lgr(nlay, nrow, ncol, delr, delc, top, botm, idomain, ncpp, ncppl)

cnlay, cnrow, cncol = lgr.get_shape()
cdelr, cdelc = lgr.get_delr_delc()
ctop, cbotm = lgr.get_top_botm()
cxorigin, cyorigin = lgr.get_lower_left()
cidomain = lgr.get_idomain()
exchangedata = lgr.get_exchange_data(angldegx=True, cdist=True)
nexg = len(exchangedata)

## Build the parent and child models and run

Build both models in one simulation: the parent GWF model and its packages, then the child GWF model and its packages, then a **GWF–GWF exchange** linking them along the refined boundary with the `exchangedata` from the `Lgr` helper. Write the input files and run MODFLOW 6.

In [ ]:
# create simulation
sim = flopy.mf6.MFSimulation(
    sim_name=model_name,
    version="mf6",
    exe_name="mf6",
    sim_ws=model_ws,
)

# create tdis package
tdis = flopy.mf6.ModflowTdis(
    sim,
)

# create gwf model
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname=model_name,
    model_nam_file=f"{model_name}.name",
)
gwf.name_file.save_flows = True

# create iterative model solution and register the gwf model with it
ims = flopy.mf6.ModflowIms(
    sim,
)

# dis
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
    idomain=idomain,
)

# initial conditions
ic = flopy.mf6.ModflowGwfic(
    gwf,
    pname="ic",
    strt=320.0,
)

# node property flow
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    icelltype=[1, 0, 0],
    k=[hk0, hk1, hk2],
    k33=[vk0, vk1, vk2],
)

# rch
rch = flopy.mf6.ModflowGwfrcha(
    gwf,
    recharge=rech,
    auxiliary=[("iface",)],
    aux={0: [6]},
)
# riv
riv = flopy.mf6.ModflowGwfriv(
    gwf,
    stress_period_data=rivspd,
)

# output control
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    pname="oc",
    budget_filerecord=f"{model_name}.cbc",
    head_filerecord=f"{model_name}.hds",
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

# create child gwf model
cmodel_name = "child"
cgwf = flopy.mf6.ModflowGwf(
    sim,
    modelname=cmodel_name,
    model_nam_file=f"{cmodel_name}.name",
)
cgwf.name_file.save_flows = True
cdis = flopy.mf6.ModflowGwfdis(
    cgwf,
    nlay=cnlay,
    nrow=cnrow,
    ncol=cncol,
    delr=cdelr,
    delc=cdelc,
    top=ctop,
    botm=cbotm,
    idomain=cidomain,
    xorigin=cxorigin,
    yorigin=cyorigin,
)
cic = flopy.mf6.ModflowGwfic(
    cgwf,
    pname="ic",
    strt=320.0,
)
cnpf = flopy.mf6.ModflowGwfnpf(
    cgwf,
    save_specific_discharge=True,
    icelltype=[1, 0, 0],
    k=[hk0, hk1, hk2],
    k33=[vk0, vk1, vk2],
)
# rch
rch = flopy.mf6.ModflowGwfrcha(
    cgwf,
    recharge=rech,
    auxiliary=[("iface",)],
    aux={0: [6]},
)

wi, wj = cgwf.modelgrid.intersect(5000, 5250)
welspd = [[(cnlay - 1, wi, wj), well_discharge]]

well = flopy.mf6.ModflowGwfwel(
    cgwf,
    print_input=True,
    stress_period_data=welspd,
)
oc = flopy.mf6.ModflowGwfoc(
    cgwf,
    pname="oc",
    budget_filerecord=f"{cmodel_name}.cbc",
    head_filerecord=f"{cmodel_name}.hds",
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

# exchange information
gwfe = flopy.mf6.ModflowGwfgwf(
    sim,
    exgtype="GWF6-GWF6",
    nexg=nexg,
    exgmnamea=gwf.name,
    exgmnameb=cgwf.name,
    exchangedata=exchangedata,
    print_flows=True,
    auxiliary=["ANGLDEGX", "CDIST"],
    filename="parent-child.gwfgwf",
)

sim.write_simulation()
success, buff = sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

## Plot the parent and child grids

Overlay the two grids to confirm the refinement. Plot the parent grid with `flopy.plot.PlotMapView().plot_grid()`, then plot the child grid on the same axes.

In [ ]:
# plot grids
fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(1, 1, 1, aspect="equal")
ilay = 0

# plot parent information
pmv = flopy.plot.PlotMapView(
    model=gwf,
    layer=ilay,
)
pmv.plot_grid()

# plot child information
pmv = flopy.plot.PlotMapView(
    model=cgwf,
    layer=ilay,
)
pmv.plot_grid()

**What to look for.** The fine child grid fills the block cut out of the coarse parent grid, with three child cells spanning each parent cell. The two grids meet cleanly along the boundary where the GWF–GWF exchange transfers flow.

## Plot the parent and child results

Map the simulated heads in the bottom layer (`ilay = 2`) for both models on the same axes and color scale, and overlay the flow directions with `.plot_vector()`. The parent heads are masked where the child grid takes over (`idomain == 0`), and the child heads are masked at inactive or dry cells (`head == 1e30`).

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(1, 1, 1, aspect="equal")
ilay = 2
plot_vectors = True

# plot parent information
pmv = flopy.plot.PlotMapView(
    model=gwf,
    layer=ilay,
)
head = gwf.output.head().get_data()
head = np.ma.masked_where(idomain == 0, head)
vmin = head[ilay].min()
vmax = head[ilay].max()
pmv.plot_array(head, vmin=vmin, vmax=vmax)
if plot_vectors:
    spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[-1]
    qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(
        spdis,
        gwf,
    )
    pmv.plot_vector(qx, qy, normalize=True, color="white")

# plot child information
pmv = flopy.plot.PlotMapView(
    model=cgwf,
    layer=ilay,
)
head = cgwf.output.head().get_data()
head = np.ma.masked_where(head == 1e30, head)
img = pmv.plot_array(head, vmin=vmin, vmax=vmax)
if plot_vectors:
    spdis = cgwf.output.budget().get_data(text="DATA-SPDIS")[-1]
    qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(
        spdis,
        cgwf,
    )
    pmv.plot_vector(qx, qy, normalize=True, color="white")

ax.set_title(f"Simulation Results for layer {ilay + 1}")
plt.colorbar(img, ax=ax, shrink=0.5)

**What to look for.** Heads vary smoothly across the parent–child boundary — no discontinuity — which shows the exchange couples the two grids correctly. The refined child grid resolves the steeper gradients and the flow field near the pumping well at the center of the domain.

## Recap

In this notebook you:

- Marked a refinement block in the parent grid with an IDOMAIN array.
- Used FloPy's `Lgr` utility to derive the child grid and the parent–child exchange data.
- Built the parent and child GWF models in one simulation and linked them with a GWF–GWF exchange.
- Ran the coupled model and confirmed heads are continuous across the refined boundary.